# H-06: SLA Threshold Compliance

**Question:** Does the agent meet the SLA thresholds defined in the ground truth for each fault type?

| | |
|---|---|
| **H₀ (Null):** | The agent's true median performance does NOT meet the SLA. |
| **Hₐ (Alternative):** | The agent's true median performance IS within the SLA. |
| **Certification rule:** | Per-sub-fault SLA compliance, rolled up to category and overall. |
| **Metrics:** | time_to_detect, time_to_mitigate |
| **Methods:** | Wilcoxon signed-rank (primary), Bootstrap BCa CI, TOST equivalence |

### Per-Sub-Fault Verdict Logic
| Condition | Verdict |
|-----------|--------|
| Wilcoxon p < α AND CI upper ≤ SLA | ✅ PASS |
| Median > SLA AND Wilcoxon not significant | ❌ FAIL |
| CI contains SLA threshold | ⚠️ CONDITIONAL |
| No SLA threshold in ground truth | ❓ NO_SLA_DEFINED |

### Category Rollup
| Condition | Category Verdict |
|-----------|------------------|
| All sub-faults PASS | PASS |
| Any sub-fault FAIL | FAIL |
| Any NO_SLA_DEFINED (none FAIL) | INCOMPLETE |
| Mix of PASS / CONDITIONAL | CONDITIONAL |

### SLA Thresholds (from ground truth)
Each sub-fault has its own SLA thresholds defined in `data/groundtruth/kubernetes/<fault>/ground_truth.yaml`.


In [1]:
import sys, os
import json
import yaml
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"
gt_root = project_root / "hypothesis_framework" / "data" / "groundtruth" / "kubernetes"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True):
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        val = q.get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

def load_sla_thresholds(gt_dir, metric_key):
    """Load SLA thresholds from ground truth YAML files.
    
    Args:
        gt_dir: Path to groundtruth/kubernetes/ directory.
        metric_key: One of 'time_to_detect', 'time_to_mitigate', 'max_tool_calls'.
    
    Returns:
        Dict[str, float]: {fault_name: threshold}
    """
    thresholds = {}
    for fault_dir in sorted(gt_dir.iterdir()):
        gt_file = fault_dir / "ground_truth.yaml"
        if not gt_file.exists():
            continue
        with open(gt_file, "r", encoding="utf-8") as f:
            gt = yaml.safe_load(f)
        sla = gt.get("ground_truth", {}).get("sla", {})
        metric_sla = sla.get(metric_key, {})
        if "threshold" in metric_sla:
            thresholds[fault_dir.name] = float(metric_sla["threshold"])
    return thresholds

# Load runs
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

# Load SLA thresholds
ttd_sla = load_sla_thresholds(gt_root, "time_to_detect")
ttm_sla = load_sla_thresholds(gt_root, "time_to_mitigate")

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r['quantitative'].get('fault_detected') == 'Yes')
    faults = sorted(set(r['fault_name'] for r in runs))
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in faults:
        fn_det = sum(1 for r in runs if r['fault_name'] == fn and r['quantitative'].get('fault_detected') == 'Yes')
        sla_ttd = ttd_sla.get(fn, '\u2014')
        sla_ttm = ttm_sla.get(fn, '\u2014')
        print(f"    {fn}: {fn_det} detected  (SLA: TTD={sla_ttd}s, TTM={sla_ttm}s)")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 28 detected  (SLA: TTD=—s, TTM=—s)
    pod-delete: 23 detected  (SLA: TTD=60.0s, TTM=120.0s)
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 13 detected  (SLA: TTD=60.0s, TTM=180.0s)
    pod-network-corruption: 14 detected  (SLA: TTD=240.0s, TTM=420.0s)
    pod-network-loss: 12 detected  (SLA: TTD=180.0s, TTM=360.0s)
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 25 detected  (SLA: TTD=60.0s, TTM=180.0s)
    pod-cpu-hog: 25 detected  (SLA: TTD=120.0s, TTM=300.0s)
    pod-memory-hog: 20 detected  (SLA: TTD=90.0s, TTM=240.0s)


---
## Step 1: SLA Thresholds from Ground Truth

Shows per-sub-fault SLA thresholds loaded from the ground truth YAML files.


In [2]:
print("SLA Thresholds (from ground truth)")
print("=" * 70)
print(f"{'Fault':30s} {'TTD (s)':>10s} {'TTM (s)':>10s}")
print("-" * 70)

all_faults = sorted(set(list(ttd_sla.keys()) + list(ttm_sla.keys())))
for fn in all_faults:
    ttd = f"{ttd_sla[fn]:.0f}" if fn in ttd_sla else "\u2014"
    ttm = f"{ttm_sla[fn]:.0f}" if fn in ttm_sla else "\u2014"
    # Mark faults present in our data
    in_data = any(fn in [r['fault_name'] for r in runs] for runs in categories.values())
    marker = "  \u2190 in dataset" if in_data else ""
    print(f"  {fn:28s} {ttd:>10s} {ttm:>10s}{marker}")

# Check for faults in data without SLA
data_faults = set()
for runs in categories.values():
    for r in runs:
        data_faults.add(r['fault_name'])
missing_sla = data_faults - set(ttd_sla.keys())
if missing_sla:
    print(f"\n\u26a0\ufe0f Faults in dataset WITHOUT ground truth SLA: {sorted(missing_sla)}")
    print("  These will be marked as NO_SLA_DEFINED in the test results.")


SLA Thresholds (from ground truth)
Fault                             TTD (s)    TTM (s)
----------------------------------------------------------------------
  disk-fill                            60        180  ← in dataset
  node-restart                         30        300
  pod-autoscaler                      120        180
  pod-cpu-hog                         120        300  ← in dataset
  pod-delete                           60        120  ← in dataset
  pod-dns-error                        60        180  ← in dataset
  pod-memory-hog                       90        240  ← in dataset
  pod-network-corruption              240        420  ← in dataset
  pod-network-loss                    180        360  ← in dataset
  pod-network-rate-limit              180        360

⚠️ Faults in dataset WITHOUT ground truth SLA: ['container-kill']
  These will be marked as NO_SLA_DEFINED in the test results.


---
## Step 2: Run H-06 — time_to_detect

Tests each sub-fault against its own SLA threshold for time-to-detect.


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h06_sla_threshold_compliance import run_sla_compliance_test

ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect")

h06_ttd = run_sla_compliance_test(
    ttd_data,
    sla_thresholds=ttd_sla,
    metric_name="time_to_detect",
    random_state=42,
)

print(f"H-06: {h06_ttd.hypothesis_name}")
print(f"Metric: {h06_ttd.metric_name}")
print(f"Overall: {h06_ttd.overall_assessment}")
print("=" * 90)

for c in h06_ttd.per_category:
    v_icon = "\u2705" if c.verdict == "PASS" else "\u274c" if c.verdict == "FAIL" else "\u26a0\ufe0f"
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Verdict: {v_icon} {c.verdict}  "
          f"(pass={c.n_passed}, fail={c.n_failed}, cond={c.n_conditional}, no_sla={c.n_no_sla})")
    if c.worst_sub_fault:
        print(f"    Worst sub-fault: {c.worst_sub_fault}")
    for sf in c.sub_faults:
        sf_icon = "\u2705" if sf.verdict == "PASS" else "\u274c" if sf.verdict == "FAIL" else "\u2753" if sf.verdict == "NO_SLA_DEFINED" else "\u26a0\ufe0f"
        sla_str = f"SLA={sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "no SLA"
        wp = f"{sf.wilcoxon_p:.6f}" if sf.wilcoxon_p is not None else "\u2014"
        ci = f"{sf.ci_upper:.1f}" if sf.ci_upper is not None else "\u2014"
        tost_icon = "\u2705" if sf.tost_equivalent else ("\u274c" if sf.tost_equivalent is not None else "\u2014")
        print(f"      {sf.fault_name:25s}: n={sf.n:3d}  median={sf.median:7.1f}s  {sla_str:12s}  "
              f"Wilcoxon p={wp:>10s}  CI_upper={ci:>8s}  TOST={tost_icon}  {sf_icon} {sf.verdict}")

if h06_ttd.warnings:
    print(f"\nWarnings:")
    for w in h06_ttd.warnings:
        print(f"  - {w}")


H-06: SLA Threshold Compliance
Metric: time_to_detect
Overall: sla_non_compliant

  application_fault (n=51, 2 sub-faults):
    Verdict: ❌ FAIL  (pass=0, fail=1, cond=0, no_sla=1)
    Worst sub-fault: pod-delete
      container-kill           : n= 28  median=  120.1s  no SLA        Wilcoxon p=         —  CI_upper=       —  TOST=—  ❓ NO_SLA_DEFINED
      pod-delete               : n= 23  median=  130.9s  SLA=60s       Wilcoxon p=  1.000000  CI_upper=   150.1  TOST=❌  ❌ FAIL

  network_fault (n=39, 3 sub-faults):
    Verdict: ⚠️ CONDITIONAL  (pass=0, fail=0, cond=3, no_sla=0)
    Worst sub-fault: pod-dns-error
      pod-dns-error            : n= 13  median=   60.0s  SLA=60s       Wilcoxon p=  0.792847  CI_upper=   279.5  TOST=❌  ⚠️ CONDITIONAL
      pod-network-corruption   : n= 14  median=  136.3s  SLA=240s      Wilcoxon p=  0.548401  CI_upper=   534.8  TOST=❌  ⚠️ CONDITIONAL
      pod-network-loss         : n= 12  median=   77.0s  SLA=180s      Wilcoxon p=  0.338623  CI_upper=   323.9 

---
## Step 3: Run H-06 — time_to_mitigate

Tests each sub-fault against its own SLA threshold for time-to-mitigate.


In [4]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate")

h06_ttm = run_sla_compliance_test(
    ttm_data,
    sla_thresholds=ttm_sla,
    metric_name="time_to_mitigate",
    random_state=42,
)

print(f"H-06: {h06_ttm.metric_name}  |  {h06_ttm.overall_assessment}")
print("=" * 90)
for c in h06_ttm.per_category:
    v_icon = '\u2705' if c.verdict == 'PASS' else '\u274c' if c.verdict == 'FAIL' else '\u26a0\ufe0f'
    print(f"  {c.category:20s}: {v_icon} {c.verdict:12s}  "
          f"(pass={c.n_passed}, fail={c.n_failed}, cond={c.n_conditional}, no_sla={c.n_no_sla})")
    for sf in c.sub_faults:
        sf_icon = '\u2705' if sf.verdict == 'PASS' else '\u274c' if sf.verdict == 'FAIL' else '\u2753' if sf.verdict == 'NO_SLA_DEFINED' else '\u26a0\ufe0f'
        sla_str = f"SLA={sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "no SLA"
        print(f"    {sf.fault_name:25s}: median={sf.median:7.1f}s  {sla_str:12s}  {sf_icon} {sf.verdict}")

if h06_ttm.warnings:
    print(f"\nWarnings: {h06_ttm.warnings}")


H-06: time_to_mitigate  |  sla_non_compliant
  application_fault   : ❌ FAIL          (pass=0, fail=1, cond=0, no_sla=1)
    container-kill           : median=  297.0s  no SLA        ❓ NO_SLA_DEFINED
    pod-delete               : median=  327.5s  SLA=120s      ❌ FAIL
  network_fault       : ❌ FAIL          (pass=0, fail=3, cond=0, no_sla=0)
    pod-dns-error            : median=  321.9s  SLA=180s      ❌ FAIL
    pod-network-corruption   : median=  521.4s  SLA=420s      ❌ FAIL
    pod-network-loss         : median=  429.8s  SLA=360s      ❌ FAIL
  resource_fault      : ❌ FAIL          (pass=0, fail=3, cond=0, no_sla=0)
    disk-fill                : median=  437.2s  SLA=180s      ❌ FAIL
    pod-cpu-hog              : median=  499.8s  SLA=300s      ❌ FAIL
    pod-memory-hog           : median=  576.6s  SLA=240s      ❌ FAIL

Warnings: ['application_fault/container-kill: no SLA threshold defined — skipping tests.']


---
## Step 4: Summary & Interpretation


In [5]:
print("H-06 SLA Threshold Compliance \u2014 Summary")
print("=" * 90)

results = {
    "time_to_detect": h06_ttd,
    "time_to_mitigate": h06_ttm,
}

for metric, r in results.items():
    overall_icon = '\u2705' if r.overall_assessment == 'sla_compliant' else '\u274c' if r.overall_assessment == 'sla_non_compliant' else '\u26a0\ufe0f'
    print(f"\n  {metric}: {overall_icon} {r.overall_assessment}")
    for c in r.per_category:
        v_icon = '\u2705' if c.verdict == 'PASS' else '\u274c' if c.verdict == 'FAIL' else '\u26a0\ufe0f'
        print(f"    {c.category:20s}: {v_icon} {c.verdict:12s}  "
              f"({c.n_passed}P/{c.n_failed}F/{c.n_conditional}C/{c.n_no_sla}U)")

print("\n" + "=" * 90)
print("\nInterpretation:")
print("  - PASS: Wilcoxon confirms median below SLA AND bootstrap CI upper ≤ SLA.")
print("  - FAIL: Median exceeds SLA and Wilcoxon cannot reject H₀.")
print("  - CONDITIONAL: CI straddles SLA boundary — more data needed.")
print("  - INCOMPLETE: Some sub-faults lack ground truth SLA thresholds.")
print("  - NO_SLA_DEFINED: Fault type has no SLA in ground truth (e.g., container-kill).")
print("  - P/F/C/U = Pass/Fail/Conditional/Unassessed sub-faults.")


H-06 SLA Threshold Compliance — Summary

  time_to_detect: ❌ sla_non_compliant
    application_fault   : ❌ FAIL          (0P/1F/0C/1U)
    network_fault       : ⚠️ CONDITIONAL   (0P/0F/3C/0U)
    resource_fault      : ❌ FAIL          (0P/3F/0C/0U)

  time_to_mitigate: ❌ sla_non_compliant
    application_fault   : ❌ FAIL          (0P/1F/0C/1U)
    network_fault       : ❌ FAIL          (0P/3F/0C/0U)
    resource_fault      : ❌ FAIL          (0P/3F/0C/0U)


Interpretation:
  - PASS: Wilcoxon confirms median below SLA AND bootstrap CI upper ≤ SLA.
  - FAIL: Median exceeds SLA and Wilcoxon cannot reject H₀.
  - CONDITIONAL: CI straddles SLA boundary — more data needed.
  - INCOMPLETE: Some sub-faults lack ground truth SLA thresholds.
  - NO_SLA_DEFINED: Fault type has no SLA in ground truth (e.g., container-kill).
  - P/F/C/U = Pass/Fail/Conditional/Unassessed sub-faults.


---
## Step 5: JSON Output


In [6]:
output = {
    "hypothesis_id": "H-06",
    "hypothesis_name": "SLA Threshold Compliance",
    "null_hypothesis": h06_ttd.null_hypothesis,
    "alt_hypothesis": h06_ttd.alt_hypothesis,
    "certification_rule": "Per-sub-fault SLA compliance using ground truth thresholds.",
    "hypothesis_metadata": {
        "methods": [
            "Wilcoxon signed-rank (primary)",
            "Bootstrap BCa CI on IQM",
            "TOST equivalence",
        ],
        "alpha": 0.05,
        "sla_source": "data/groundtruth/kubernetes/*/ground_truth.yaml",
        "sla_thresholds": {
            "time_to_detect": ttd_sla,
            "time_to_mitigate": ttm_sla,
        },
        "categories": list(categories.keys()),
        "mode": "SLA-aware (activates only with SLA thresholds)",
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "per_category": [
            {
                "category": c.category,
                "n": c.n,
                "n_sub_faults": c.n_sub_faults,
                "verdict": c.verdict,
                "n_passed": c.n_passed,
                "n_failed": c.n_failed,
                "n_conditional": c.n_conditional,
                "n_no_sla": c.n_no_sla,
                "worst_sub_fault": c.worst_sub_fault,
                "sub_faults": [
                    {
                        "fault_name": sf.fault_name,
                        "n": sf.n,
                        "sla_threshold": sf.sla_threshold,
                        "median": sf.median,
                        "wilcoxon_p": sf.wilcoxon_p,
                        "ci_upper": sf.ci_upper,
                        "tost_equivalent": sf.tost_equivalent,
                        "tost_p": sf.tost_p,
                        "verdict": sf.verdict,
                    }
                    for sf in c.sub_faults
                ],
            }
            for c in r.per_category
        ],
        "warnings": r.warnings,
    }

print(json.dumps(output, indent=2, default=str, ensure_ascii=False))


{
  "hypothesis_id": "H-06",
  "hypothesis_name": "SLA Threshold Compliance",
  "null_hypothesis": "The agent's true median performance does NOT meet the SLA.",
  "alt_hypothesis": "The agent's true median performance IS within the SLA.",
  "certification_rule": "Per-sub-fault SLA compliance using ground truth thresholds.",
  "hypothesis_metadata": {
    "methods": [
      "Wilcoxon signed-rank (primary)",
      "Bootstrap BCa CI on IQM",
      "TOST equivalence"
    ],
    "alpha": 0.05,
    "sla_source": "data/groundtruth/kubernetes/*/ground_truth.yaml",
    "sla_thresholds": {
      "time_to_detect": {
        "disk-fill": 60.0,
        "node-restart": 30.0,
        "pod-autoscaler": 120.0,
        "pod-cpu-hog": 120.0,
        "pod-delete": 60.0,
        "pod-dns-error": 60.0,
        "pod-memory-hog": 90.0,
        "pod-network-corruption": 240.0,
        "pod-network-loss": 180.0,
        "pod-network-rate-limit": 180.0
      },
      "time_to_mitigate": {
        "disk-fill": 18